# Qwen-Image 2.1 · ComfyUI + ComfyUI-Manager · Cloudflare link

Open [Google Colab](https://colab.research.google.com/), choose **Upload notebook → Browse**, and select this `.ipynb` file. Save your own copy in Drive and give it a recognizable name.

Run the setup cells **1–4 in order**. Step 4 stays running while you use ComfyUI; stop it before running another cell. The export cell at the end is for **after** your work:

1. **Install** ComfyUI (newest stable release) with ComfyUI-Manager.
2. **Download** the Qwen-Image 2.1 models — the right size for your GPU is picked automatically.
3. *Optional:* **download other models** (LoRAs, upscalers, …) from any direct link.
4. **Start** — you get a `https://….trycloudflare.com` link and ComfyUI opens with the **Qwen-Image 2.1 workflow already loaded**. Write a prompt and press **Run**.

Before connecting, use **Runtime → Change runtime type** to choose a GPU. Changing hardware can replace the runtime and its files, so download your work first. Use a GPU runtime: **Runtime → Change runtime type → T4 GPU** or better. Setup and image times vary with the allocated GPU, download speed and first-run model loading.

> T4 can be allocated on free accounts; A100 requires eligible paid access. Colab warns that free runtimes used primarily through an external web UI may be terminated ([FAQ](https://research.google.com/colaboratory/faq.html)); availability is not guaranteed. The link is public — don't share it. Qwen-Image 2.1 weights use the Qwen Research License (non-commercial).

## 1 · Install
ComfyUI from [github.com/Comfy-Org/ComfyUI](https://github.com/Comfy-Org/ComfyUI), its Python packages, the built-in **ComfyUI-Manager** (the **Extensions** button in ComfyUI) and **cloudflared** (makes the public link, no account needed). Colab's own PyTorch is kept.

In [ ]:
#@title 1 · Install ComfyUI + ComfyUI-Manager 
COMFYUI_VERSION = "latest stable release"  #@param ["latest stable release", "master (nightly)"]

import os, re, subprocess, sys, time

COMFY_DIR   = "/content/ComfyUI"
COMFY_REPO  = "https://github.com/Comfy-Org/ComfyUI.git"
CLOUDFLARED = "/usr/local/bin/cloudflared"

def run(cmd, cwd=None):
    """Run a command, show its output, and stop with a clear error if it fails."""
    print(">>", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print("   ", line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {proc.returncode}): {' '.join(cmd)}")

# GPU check - read with nvidia-smi so this notebook itself doesn't take any GPU memory
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError("No GPU! Runtime > Change runtime type > pick a GPU (T4, L4, A100...) > Save, then run again.")
print("GPU:", gpu, "\n")
started = time.time()

# 1) ComfyUI itself: the newest stable release (vX.Y.Z tag) from GitHub
if os.path.isfile(f"{COMFY_DIR}/main.py"):
    print("ComfyUI is already installed - skipping the download.")
elif COMFYUI_VERSION.startswith("latest"):
    tags = subprocess.run(["git", "ls-remote", "--tags", "--refs", COMFY_REPO],
                          capture_output=True, text=True, check=True).stdout
    latest = "v" + ".".join(max(re.findall(r"refs/tags/v(\d+)\.(\d+)\.(\d+)$", tags, re.M),
                                key=lambda v: tuple(map(int, v))))
    print("Newest stable ComfyUI release:", latest)
    run(["git", "-c", "advice.detachedHead=false", "clone", "--depth", "1", "--branch", latest, COMFY_REPO, COMFY_DIR])
else:
    run(["git", "clone", "--depth", "1", COMFY_REPO, COMFY_DIR])

# 2) Python packages of ComfyUI + ComfyUI-Manager (Colab's own CUDA PyTorch is kept - torch is not pinned)
print("\nInstalling Python packages...")
run([sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off",
     "-r", "requirements.txt", "-r", "manager_requirements.txt"], cwd=COMFY_DIR)

# 3) cloudflared: creates the public https link in step 4
if not os.path.exists(CLOUDFLARED):
    run(["wget", "-q", "-O", CLOUDFLARED,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    os.chmod(CLOUDFLARED, 0o755)

from importlib.metadata import version
with open(f"{COMFY_DIR}/comfyui_version.py") as f:
    comfy_version = re.search(r'__version__ = "([^"]+)"', f.read()).group(1)
print(f"\nDone in {time.time() - started:.0f} s:  ComfyUI {comfy_version}  |  "
      f"ComfyUI-Manager {version('comfyui-manager')}  |  PyTorch {version('torch')}")

## 2 · Download the Qwen-Image 2.1 models
Qwen-Image 2.1 is three files from the official ComfyUI repack **[Comfy-Org/Qwen-Image-2.1](https://huggingface.co/Comfy-Org/Qwen-Image-2.1)**:

| File | Job | Saved to |
|---|---|---|
| Diffusion model (7B) | paints the image | `ComfyUI/models/diffusion_models` |
| Text encoder (Qwen3-VL-8B) | reads your prompt | `ComfyUI/models/text_encoders` |
| VAE | turns the result into pixels | `ComfyUI/models/vae` |

`MODEL_PRECISION = auto` checks **both GPU VRAM and system RAM**:

| Memory available | Files | Download |
|---|---|---|
| either VRAM or system RAM below 20 GiB (typical T4) | int8 diffusion model + **w4a8** text encoder | 14.2 GB |
| both VRAM and system RAM at least 20 GiB | int8 diffusion model + int8 text encoder — the official template's files | 17.3 GB |
| *pick it yourself* | bf16 + bf16, full precision (40 GB+ GPUs) | 32.4 GB |

int8 and w4a8 are compressed copies (8-bit / 4-bit weights) that run natively in ComfyUI. Quantization reduces memory use and can change output; compare your own task if that matters. Download sizes above are decimal GB; memory thresholds use GiB. Running the cell again skips finished files and resumes broken downloads.

In [ ]:
#@title 2 · Download the Qwen-Image 2.1 models 
# Plain output also works when Colab rich-output widgets cannot load.
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
#@markdown **auto** picks the right files for your GPU (see the table above).
MODEL_PRECISION = "auto"  #@param ["auto", "int8 + w4a8 text encoder (14.2 GB) - T4 / 16 GB GPUs", "int8 + int8 text encoder (17.3 GB) - official template, 24 GB+ GPUs", "bf16 + bf16 text encoder (32.4 GB) - full precision, 40 GB+ GPUs"]
#@markdown **Prompt enhancer** - only for ComfyUI's *official* Qwen-Image 2.1 templates. The workflow this notebook opens doesn't use it.
PROMPT_ENHANCER = "none"  #@param ["none", "text-to-image (+9.47 GB)", "image edit (+9.47 GB)", "both (+18.9 GB)"]
#@markdown **Hugging Face token** - not needed, these files are public.
HF_TOKEN = ""  #@param {type:"string"}

import json, os, shutil, subprocess, time
import psutil

MODELS_DIR = "/content/ComfyUI/models"
REPO_ID    = "Comfy-Org/Qwen-Image-2.1"        # https://huggingface.co/Comfy-Org/Qwen-Image-2.1

# (path in the Hugging Face repo, exact size in bytes, what it is). The repo uses ComfyUI's folder
# names, so "vae/x.safetensors" is saved as ComfyUI/models/vae/x.safetensors - right where ComfyUI looks.
FILES = {
    "dit_int8": ("diffusion_models/qwen_image_2.1_int8_convrot.safetensors",   7_256_783_064, "diffusion model (7B), int8"),
    "dit_bf16": ("diffusion_models/qwen_image_2.1_bf16.safetensors",           14_230_280_616, "diffusion model (7B), bf16"),
    "te_w4a8":  ("text_encoders/qwen3vl_8b_w4a8.safetensors",                   6_312_105_364, "text encoder Qwen3-VL-8B, w4a8"),
    "te_int8":  ("text_encoders/qwen3vl_8b_int8_convrot.safetensors",           9_350_798_360, "text encoder Qwen3-VL-8B, int8"),
    "te_bf16":  ("text_encoders/qwen3vl_8b_bf16.safetensors",                  17_534_334_616, "text encoder Qwen3-VL-8B, bf16"),
    "vae":      ("vae/qwen_image_2.1_vae_bf16.safetensors",                       675_509_688, "VAE"),
    "pe_t2i":   ("text_encoders/qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors", 9_471_072_252, "prompt enhancer, text-to-image"),
    "pe_i2i":   ("text_encoders/qwen3.5_9b_qwen_image_2.1_pe_i2i.int8_convrot.safetensors", 9_471_072_252, "prompt enhancer, image edit"),
}
PRESETS = {"int8_w4a8": ["dit_int8", "te_w4a8", "vae"],     # diffusion model, text encoder, VAE
           "int8":      ["dit_int8", "te_int8", "vae"],
           "bf16":      ["dit_bf16", "te_bf16", "vae"]}

# 1) Which files? "auto" looks at the GPU memory (VRAM) and the system RAM
if MODEL_PRECISION == "auto":
    smi = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.split()
    vram_gb = int(smi[0]) / 1024 if smi else 0
    ram_gb = psutil.virtual_memory().total / 1024**3
    preset = "int8" if vram_gb >= 20 and ram_gb >= 20 else "int8_w4a8"
    print(f"auto: {vram_gb:.0f} GB VRAM + {ram_gb:.0f} GB RAM  ->  {preset}")
elif "w4a8" in MODEL_PRECISION:
    preset = "int8_w4a8"
elif MODEL_PRECISION.startswith("bf16"):
    preset = "bf16"
else:
    preset = "int8"
enhancers = {"none": [], "text-to-image": ["pe_t2i"], "image edit": ["pe_i2i"], "both": ["pe_t2i", "pe_i2i"]}
keys = PRESETS[preset] + enhancers[PROMPT_ENHANCER.split(" (")[0]]

def is_downloaded(key):
    path, size, _ = FILES[key]
    return os.path.isfile(f"{MODELS_DIR}/{path}") and os.path.getsize(f"{MODELS_DIR}/{path}") == size

missing = [k for k in keys if not is_downloaded(k)]
need = sum(FILES[k][1] for k in missing)
if need + 2e9 > shutil.disk_usage("/content").free:
    raise RuntimeError("Not enough free disk space.")
print(f"{len(keys)} files, {sum(FILES[k][1] for k in keys) / 1e9:.1f} GB  ({need / 1e9:.1f} GB still to download)\n")

# 2) Download with hf_xet - Hugging Face's fast downloader (resumes if interrupted)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()                    # hide the harmless "unauthenticated requests" warning

for i, k in enumerate(keys, 1):
    path, size, what = FILES[k]
    print(f"[{i}/{len(keys)}] {what}  ({size / 1e9:.2f} GB)  ->  ComfyUI/models/{path}")
    if k not in missing:
        print("      already downloaded")
        continue
    t0 = time.time()
    hf_hub_download(repo_id=REPO_ID, filename=path, local_dir=MODELS_DIR, token=HF_TOKEN.strip() or None)
    if not is_downloaded(k):
        raise RuntimeError(f"{path} is incomplete - run this cell again to resume.")
    print(f"      done in {time.time() - t0:.0f} s")

# 3) Remember the choice - step 4 builds the workflow with these files
dit, te, vae = (os.path.basename(FILES[k][0]) for k in PRESETS[preset])
with open("/content/qwen_image_2.1_selection.json", "w") as f:
    json.dump({"diffusion_model": dit, "text_encoder": te, "vae": vae}, f, indent=2)
print(f"\nReady. The workflow will use: {dit} + {te} + {vae}")

## 3 · *Optional:* download other models
Paste up to four **direct download links**, pick a folder for each, run. On Hugging Face open the file → right-click the download button (↓) → **Copy link address** (`…/resolve/main/…`). A `/blob/` page link works too. **auto** reads the folder from the link (works for all `Comfy-Org` repos). The first row is a small upscaler example; replace or clear it. The other rows are empty.

Folders: `diffusion_models` main models · `text_encoders` · `vae` · `loras` · `checkpoints` all-in-one models · `model_patches` model-patch weights · `controlnet` conditioning-side control weights · `upscale_models` · `workflows` .json files for the Workflows sidebar.

| More example links | Folder | Size |
|---|---|---|
| `https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_bf16.safetensors` | auto | 14.2 GB |
| `https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors` | auto | 9.35 GB |
| `https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/model_patches/qwen_image_2.1_fun_controlnet_union_int8_convrot.safetensors` | auto | 3.78 GB |
| `https://huggingface.co/Comfy-Org/Real-ESRGAN_repackaged/resolve/main/RealESRGAN_x4plus.safetensors` | upscale_models | 67 MB |

To use a **LoRA**, add a *LoraLoaderModelOnly* node after *Load Diffusion Model*. Use only a LoRA trained for your exact base family, with its documented sampler settings. Downloading a file does not make it compatible with the current graph. An **upscaler** goes into *Load Upscale Model* + *Upscale Image (using Model)* after *VAE Decode*.
For Lecture 6 structural control, use the separately matched base, control weights, VAE and preprocessor in the lesson materials. The original Qwen-Image InstantX Union is **not** a plug-in for the Qwen-Image 2.1 starter. Pick the destination specified by that workflow; `auto` detects path names, not compatibility.


In [ ]:
#@title 3 · (Optional) Download other models from direct links
# Plain output also works when Colab rich-output widgets cannot load.
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
#@markdown Paste a **direct download link** and choose its folder. Empty rows and files you already have are skipped. Run this **before step 4** (if step 4 is running, stop it with ■ first).
#@markdown <br>Row 1 is a small upscaler example; replace or clear it. Rows 2-4 are empty. Check the model family and required loader in your workflow.
#@markdown <br>**File 1** - link:
LINK_1 = "https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.safetensors"  #@param {type:"string"}
#@markdown ↳ save into folder:
FOLDER_1 = "upscale_models"  #@param ["auto (read it from the link)", "diffusion_models", "text_encoders", "vae", "loras", "checkpoints", "model_patches", "controlnet", "upscale_models", "latent_upscale_models", "clip_vision", "embeddings", "style_models", "vae_approx", "audio_encoders", "workflows (ComfyUI .json workflows)"]
#@markdown **File 2** - link:
LINK_2 = ""  #@param {type:"string"}
#@markdown ↳ save into folder:
FOLDER_2 = "loras"  #@param ["auto (read it from the link)", "diffusion_models", "text_encoders", "vae", "loras", "checkpoints", "model_patches", "controlnet", "upscale_models", "latent_upscale_models", "clip_vision", "embeddings", "style_models", "vae_approx", "audio_encoders", "workflows (ComfyUI .json workflows)"]
#@markdown **File 3** - link:
LINK_3 = ""  #@param {type:"string"}
#@markdown ↳ save into folder:
FOLDER_3 = "auto (read it from the link)"  #@param ["auto (read it from the link)", "diffusion_models", "text_encoders", "vae", "loras", "checkpoints", "model_patches", "controlnet", "upscale_models", "latent_upscale_models", "clip_vision", "embeddings", "style_models", "vae_approx", "audio_encoders", "workflows (ComfyUI .json workflows)"]
#@markdown **File 4** - link:
LINK_4 = ""  #@param {type:"string"}
#@markdown ↳ save into folder:
FOLDER_4 = "auto (read it from the link)"  #@param ["auto (read it from the link)", "diffusion_models", "text_encoders", "vae", "loras", "checkpoints", "model_patches", "controlnet", "upscale_models", "latent_upscale_models", "clip_vision", "embeddings", "style_models", "vae_approx", "audio_encoders", "workflows (ComfyUI .json workflows)"]
#@markdown **Hugging Face token** - only needed for gated or private repos (accept the license on the model page first).
HF_TOKEN = ""  #@param {type:"string"}

import os, re, shutil, urllib.parse

MODELS_DIR    = "/content/ComfyUI/models"
WORKFLOWS_DIR = "/content/ComfyUI/user/default/workflows"
STAGING_DIR   = "/content/.downloads"          # files are downloaded here first, then moved
MODEL_FOLDERS = ["diffusion_models", "text_encoders", "vae", "loras", "checkpoints", "model_patches",
                 "controlnet", "upscale_models", "latent_upscale_models", "clip_vision", "embeddings",
                 "style_models", "vae_approx", "audio_encoders"]
FOLDER_ALIASES = {"unet": "diffusion_models", "clip": "text_encoders"}   # old folder names

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
import requests
from huggingface_hub import get_hf_file_metadata, hf_hub_download, hf_hub_url
from huggingface_hub.utils import logging as hf_logging
from tqdm import tqdm
hf_logging.set_verbosity_error()
token = HF_TOKEN.strip() or None


def target_folder(choice, link):
    """Turn the dropdown choice into a real folder. 'auto' looks for a ComfyUI folder name in the link."""
    if choice.startswith("workflows"):
        return WORKFLOWS_DIR
    if not choice.startswith("auto"):
        return f"{MODELS_DIR}/{choice}"
    path_parts = urllib.parse.urlparse(link).path.split("/")[:-1]
    for part in reversed(path_parts):                       # the folder closest to the file name wins
        name = FOLDER_ALIASES.get(part, part)
        if name in MODEL_FOLDERS:
            return f"{MODELS_DIR}/{name}"
    if link.split("?")[0].lower().endswith(".json"):
        return WORKFLOWS_DIR
    raise ValueError("'auto' found no ComfyUI folder name in this link - pick the folder in the dropdown.")


def parse_hf_link(link):
    """https://huggingface.co/<user>/<repo>/resolve|blob/<branch>/<path>  ->  parts, or None if not a Hugging Face file."""
    m = re.match(r"https?://(?:www\.)?(?:huggingface\.co|hf\.co)/(?:(datasets|spaces)/)?([^/]+/[^/]+)/(?:resolve|blob)/([^/]+)/(.+)",
                 link.split("?")[0].split("#")[0])
    if not m:
        return None
    kind, repo_id, revision, path = m.groups()
    return {"repo_type": {"datasets": "dataset", "spaces": "space"}.get(kind, "model"),
            "repo_id": repo_id, "revision": revision, "path": urllib.parse.unquote(path)}


def human(size):
    if not size:
        return "size unknown"
    return f"{size / 1e9:.2f} GB" if size >= 1e9 else f"{size / 1e6:.0f} MB" if size >= 1e6 else f"{size / 1e3:.0f} KB"


def have_file(path, size):
    return os.path.isfile(path) and (size is None or os.path.getsize(path) == size)


def check_disk(size):
    if size and size + 1e9 > shutil.disk_usage("/content").free:
        raise RuntimeError(f"not enough free disk space for {human(size)}")


def download_from_hugging_face(hf, folder):
    name = os.path.basename(hf["path"])
    url = hf_hub_url(hf["repo_id"], hf["path"], repo_type=hf["repo_type"], revision=hf["revision"])
    size = get_hf_file_metadata(url, token=token).size        # also checks that the link works
    dest = os.path.join(folder, name)
    if have_file(dest, size):
        return dest, size, "already there"
    check_disk(size)
    print(f"  downloading {name} ({human(size)}) from huggingface.co/{hf['repo_id']} ...")
    tmp = hf_hub_download(hf["repo_id"], hf["path"], repo_type=hf["repo_type"], revision=hf["revision"],
                          local_dir=STAGING_DIR, token=token)
    shutil.move(tmp, dest)
    return dest, size, "downloaded"


def download_from_other_site(link, folder):
    """Any other direct link (GitHub releases, your own server...)."""
    with requests.get(link, stream=True, timeout=60, allow_redirects=True,
                      headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "identity"}) as r:
        r.raise_for_status()
        cd = r.headers.get("Content-Disposition", "")
        m = re.search(r"filename\*=UTF-8''([^;]+)", cd) or re.search(r'filename="?([^";]+)"?', cd)
        name = urllib.parse.unquote(m.group(1)) if m else urllib.parse.unquote(urllib.parse.urlparse(r.url).path.split("/")[-1])
        name = os.path.basename(name.strip()) or "downloaded_file"
        size = int(r.headers["Content-Length"]) if r.headers.get("Content-Length", "").isdigit() else None
        if r.headers.get("Content-Encoding", "identity") != "identity":
            size = None                    # compressed transfer: Content-Length is not the real file size
        dest = os.path.join(folder, name)
        if have_file(dest, size):
            return dest, size, "already there"
        check_disk(size)
        tmp = os.path.join(STAGING_DIR, name + ".part")
        with open(tmp, "wb") as f, tqdm(total=size, unit="B", unit_scale=True, desc=name) as bar:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))
    if size and os.path.getsize(tmp) != size:
        raise RuntimeError("the download was cut off - run the cell again")
    shutil.move(tmp, dest)
    return dest, os.path.getsize(dest), "downloaded"


def friendly(error, is_hf):
    """Short, human error message."""
    text = str(error)
    if any(s in text.lower() for s in ("401", "403", "gated", "unauthorized", "forbidden")):
        return ("access denied - this is a gated or private file: accept its license on Hugging Face and paste "
                "your HF_TOKEN above" if is_hf else "access denied - this site needs a login or API key for the file")
    if any(s in text.lower() for s in ("404", "entrynotfound", "repositorynotfound", "not found")):
        return "not found - check the link (copy it again from the file's download button)"
    return text.splitlines()[0][:300] if text else type(error).__name__


if not os.path.isfile("/content/ComfyUI/main.py"):
    raise RuntimeError("Install ComfyUI first (step 1), then run this cell.")

rows = [(LINK_1, FOLDER_1), (LINK_2, FOLDER_2), (LINK_3, FOLDER_3), (LINK_4, FOLDER_4)]
os.makedirs(STAGING_DIR, exist_ok=True)
results = []
for i, (link, choice) in enumerate(rows, 1):
    link = link.strip()
    if not link:
        continue
    print(f"[{i}] {link}")
    is_hf = bool(re.match(r"https?://(?:www\.)?(?:huggingface\.co|hf\.co)/", link))
    try:
        if not link.startswith(("http://", "https://")):
            raise ValueError("this is not a web link - it must start with https://")
        hf = parse_hf_link(link)
        if is_hf and hf is None:
            raise ValueError("this is a Hugging Face page, not a file - open the file and copy its download link "
                             "(it contains /resolve/)")
        folder = target_folder(choice, link)
        os.makedirs(folder, exist_ok=True)
        print(f"  folder: {folder}")
        dest, size, status = download_from_hugging_face(hf, folder) if hf else download_from_other_site(link, folder)
        print(f"  {status}: {dest}  ({human(size)})\n")
        results.append(("OK", dest))
    except Exception as e:
        print(f"  FAILED: {friendly(e, is_hf)}\n")
        results.append(("FAILED", link))

if not results:
    print("Nothing to do - paste a link into one of the LINK boxes above.")
else:
    print("Summary:")
    for status, what in results:
        print(f"  {status:6s} {what}")
    print("\nThe files appear in ComfyUI's loader dropdowns when you start step 4.")

## 4 · Start ComfyUI
Creates the Cloudflare link, starts ComfyUI with ComfyUI-Manager and opens the **Qwen-Image 2.1 text-to-image workflow** for you (euler · simple · 25 steps · CFG 1 · 1024×1024 — the official settings). It is also saved in the Workflows sidebar (press **W**).

**Keep this cell running** — it shows ComfyUI's live log. Press ■ to stop ComfyUI; run the cell again to restart it with a new link.

In [ ]:
#@title 4 · Start ComfyUI (Cloudflare link) - the Qwen-Image 2.1 workflow opens automatically
#@markdown Keep this cell running while you work - it shows ComfyUI's live log. Press ■ to stop ComfyUI; run it again for a new link.

import json, os, re, signal, subprocess, sys, time, urllib.request, uuid

COMFY_DIR   = "/content/ComfyUI"
MODELS_DIR  = f"{COMFY_DIR}/models"
CLOUDFLARED = "/usr/local/bin/cloudflared"
PORT        = 8188
LOCAL_URL   = f"http://127.0.0.1:{PORT}"
COMFY_LOG   = "/content/comfyui.log"
TUNNEL_LOG  = "/content/cloudflared.log"
TEMPLATE_SOURCE, TEMPLATE_NAME = "qwen_image_21_colab", "Qwen-Image-2.1_Text-to-Image"
HF_FILES = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main"

if not os.path.isfile(f"{COMFY_DIR}/main.py"):
    raise RuntimeError("Run step 1 first (install ComfyUI).")

# ---------------------------------------------------------------------------------------------
# 4.1  The Qwen-Image 2.1 text-to-image workflow (same nodes and settings as ComfyUI's official
#      template: euler / simple, 25 steps, CFG 1, 1024x1024) using the model files you downloaded
# ---------------------------------------------------------------------------------------------
def pick(folder, preferred, prefix):
    """Use the file chosen in step 2; otherwise any matching file found in the folder."""
    names = sorted(os.listdir(f"{MODELS_DIR}/{folder}")) if os.path.isdir(f"{MODELS_DIR}/{folder}") else []
    if preferred in names:
        return preferred
    return next((n for n in names if n.startswith(prefix) and n.endswith(".safetensors")), preferred)

chosen = {}
if os.path.exists("/content/qwen_image_2.1_selection.json"):          # written by step 2
    with open("/content/qwen_image_2.1_selection.json") as f:
        chosen = json.load(f)
dit = pick("diffusion_models", chosen.get("diffusion_model", "qwen_image_2.1_int8_convrot.safetensors"), "qwen_image_2.1_")
te  = pick("text_encoders", chosen.get("text_encoder", "qwen3vl_8b_int8_convrot.safetensors"), "qwen3vl_8b_")
vae = pick("vae", chosen.get("vae", "qwen_image_2.1_vae_bf16.safetensors"), "qwen_image_2.1_vae")

def node(node_id, kind, pos, size, widgets, inputs=(), outputs=(), order=0, title=None, model=None):
    n = {"id": node_id, "type": kind, "pos": pos, "size": size, "flags": {}, "order": order, "mode": 0,
         "inputs": [{"name": a, "type": b, "link": c, **({"shape": 7} if c is None else {})} for a, b, c in inputs],
         "outputs": [{"name": a, "type": b, "links": c} for a, b, c in outputs],
         "properties": {"Node name for S&R": kind}, "widgets_values": widgets}
    if title:
        n["title"] = title
    if model:   # lets ComfyUI offer a download if the file is missing
        n["properties"]["models"] = [{"name": model[1], "directory": model[0], "url": f"{HF_FILES}/{model[0]}/{model[1]}"}]
    return n

prompt = ('A cozy bookstore window on a rainy evening, warm golden light inside, stacks of old books, a hand-painted '
          'sign on the glass reads "QWEN IMAGE 2.1", raindrops and reflections on the wet street, photorealistic, 35mm photo')
note = ("## Qwen-Image 2.1 - text to image\n\n1. Write your prompt in **Text Encode Qwen Image 2.1**.\n"
        "2. Size in **Empty Latent Image**: 1024x1024 (fast) or up to 2048x2048 (native 2K, slower).\n3. Click **Run**.\n\n"
        "**Settings:** euler + simple, 25 steps (40-50 = Qwen's own pipeline), **CFG 1** - the negative prompt is only "
        "used when CFG is above 1.\n\n**Text in the image:** put it in double quotes.\n\n**Transparent PNG:** This is an "
        "RGBA format image with transparency. *your subject*. The image has an alpha channel and a transparent background.")
workflow = {
    "id": str(uuid.uuid4()), "revision": 0, "last_node_id": 18, "last_link_id": 9, "groups": [], "config": {},
    "extra": {"ds": {"scale": 0.75, "offset": [60, 140]}}, "version": 0.4,
    "nodes": [
        node(10, "UNETLoader", [0, 0], [400, 82], [dit, "default"], outputs=[("MODEL", "MODEL", [1])],
             order=0, model=("diffusion_models", dit)),
        node(12, "CLIPLoader", [0, 130], [400, 106], [te, "qwen_image", "default"], outputs=[("CLIP", "CLIP", [3])],
             order=1, model=("text_encoders", te)),
        node(13, "VAELoader", [0, 290], [400, 58], [vae], outputs=[("VAE", "VAE", [8])], order=2, model=("vae", vae)),
        node(15, "EmptyLatentImage", [0, 400], [400, 106], [1024, 1024, 1], outputs=[("LATENT", "LATENT", [6])], order=3),
        node(1, "MarkdownNote", [0, 560], [400, 420], [note], order=4, title="How to use"),
        node(11, "QwenImage21Cache", [460, 0], [320, 82], ["auto", "default"],
             inputs=[("model", "MODEL", 1)], outputs=[("MODEL", "MODEL", [2])], order=5),
        node(14, "TextEncodeQwenImage21", [460, 130], [460, 460], [prompt, "", 1024],
             inputs=[("clip", "CLIP", 3), ("images.image_1", "IMAGE", None), ("vae", "VAE", None)],
             outputs=[("positive", "CONDITIONING", [4]), ("negative", "CONDITIONING", [5]), ("latent", "LATENT", None)],
             order=6),
        node(16, "KSampler", [980, 0], [320, 474], [0, "randomize", 25, 1, "euler", "simple", 1],   # tall: room for the live preview
             inputs=[("model", "MODEL", 2), ("positive", "CONDITIONING", 4), ("negative", "CONDITIONING", 5),
                     ("latent_image", "LATENT", 6)], outputs=[("LATENT", "LATENT", [7])], order=7),
        node(17, "VAEDecode", [980, 540], [220, 46], [], inputs=[("samples", "LATENT", 7), ("vae", "VAE", 8)],
             outputs=[("IMAGE", "IMAGE", [9])], order=8),
        node(18, "SaveImage", [1360, 0], [620, 680], ["Qwen-Image-2.1/T2I"], inputs=[("images", "IMAGE", 9)], order=9),
    ],
    "links": [[1, 10, 0, 11, 0, "MODEL"], [2, 11, 0, 16, 0, "MODEL"], [3, 12, 0, 14, 0, "CLIP"],
              [4, 14, 0, 16, 1, "CONDITIONING"], [5, 14, 1, 16, 2, "CONDITIONING"], [6, 15, 0, 16, 3, "LATENT"],
              [7, 16, 0, 17, 0, "LATENT"], [8, 13, 0, 17, 1, "VAE"], [9, 17, 0, 18, 0, "IMAGE"]],
}

# Saved as a template (a tiny node-less "custom node" folder that ComfyUI lists under Templates) so the
# link below can open it automatically, and once in the Workflows sidebar (press W) for later.
template_dir = f"{COMFY_DIR}/custom_nodes/{TEMPLATE_SOURCE}"
os.makedirs(f"{template_dir}/example_workflows", exist_ok=True)
with open(f"{template_dir}/__init__.py", "w") as f:
    f.write("# Adds no nodes - only ships the Qwen-Image 2.1 workflow (see example_workflows/).\nNODE_CLASS_MAPPINGS = {}\n")
with open(f"{template_dir}/example_workflows/{TEMPLATE_NAME}.json", "w") as f:
    json.dump(workflow, f, indent=2)
user_dir = f"{COMFY_DIR}/user/default"
os.makedirs(f"{user_dir}/workflows", exist_ok=True)
if not os.path.exists(f"{user_dir}/workflows/Qwen-Image 2.1 - Text to Image.json"):    # never overwrite your edits
    with open(f"{user_dir}/workflows/Qwen-Image 2.1 - Text to Image.json", "w") as f:
        json.dump(workflow, f, indent=2)
if not os.path.exists(f"{user_dir}/comfy.settings.json"):                              # skip the first-visit tour
    with open(f"{user_dir}/comfy.settings.json", "w") as f:
        json.dump({"Comfy.TutorialCompleted": True}, f)
print(f"Workflow: {dit} + {te} + {vae}")

# ---------------------------------------------------------------------------------------------
# 4.2  Background programs: cloudflared (public link) and ComfyUI
# ---------------------------------------------------------------------------------------------
def tail(path, lines=40):
    try:
        with open(path, errors="replace") as f:
            return "".join(f.readlines()[-lines:])
    except FileNotFoundError:
        return ""

def process_record(name):
    return f"/content/.{name}.process.json"

def start(name, cmd, log_path, cwd=None):
    import psutil
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=cwd, stdout=log, stderr=subprocess.STDOUT,
                                start_new_session=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    identity = psutil.Process(proc.pid)
    record = {"pid": proc.pid, "created": identity.create_time(), "cmdline": [str(arg) for arg in cmd],
              "process_group": os.getpgid(proc.pid), "state": "running"}
    with open(process_record(name), "w") as f:
        json.dump(record, f)
    return proc

def stop(name):
    """Stop only a process launched by this cell, never a matching-name program."""
    import psutil
    record_path = process_record(name)
    if not os.path.isfile(record_path):
        return
    with open(record_path) as f:
        record = json.load(f)
    if record.get("state") == "stopped":
        return
    try:
        proc = psutil.Process(record["pid"])
        if (proc.create_time() != record["created"] or proc.cmdline() != record["cmdline"]
                or os.getpgid(proc.pid) != proc.pid or record["process_group"] != proc.pid):
            raise RuntimeError(f"{name}: process identity changed; no process was stopped.")
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=10)
        except psutil.TimeoutExpired:
            if proc.create_time() != record["created"]:
                raise RuntimeError(f"{name}: process changed while stopping.")
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait(timeout=5)
    except (psutil.NoSuchProcess, ProcessLookupError):
        pass
    record["state"] = "stopped"
    with open(record_path, "w") as f:
        json.dump(record, f)

def url_ok(url, timeout=10, headers=None):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0", **(headers or {})})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

def stop_all():
    stop("cloudflared")
    stop("comfyui")

stop_all()                                               # old copies from an earlier run of this cell
try:
    # Cloudflare tunnel first: we need its address for ComfyUI's --enable-cors-header.
    # --protocol http2: the default QUIC (UDP) tunnel is extremely slow on Colab.
    tunnel = start("cloudflared", [CLOUDFLARED, "tunnel", "--no-autoupdate", "--protocol", "http2",
                                   "--url", LOCAL_URL], TUNNEL_LOG)
    print("Creating the Cloudflare link", end="", flush=True)
    public_url, t0 = None, time.time()
    while public_url is None and time.time() - t0 < 90 and tunnel.poll() is None:
        log = tail(TUNNEL_LOG, 400)
        found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", log) if u != "https://api.trycloudflare.com"]
        if found and "Registered tunnel connection" in log:
            public_url = found[-1]
        else:
            print(".", end="", flush=True)
            time.sleep(2)
    if public_url is None:
        raise RuntimeError("Cloudflare gave no link (its free tunnels are sometimes busy) - wait a minute and run "
                           "this cell again.\n" + tail(TUNNEL_LOG, 20))
    print(" ok")

    # ComfyUI. --enable-cors-header <link>: without it, ComfyUI answers "403" to pages opened from another site
    # (like the button below). --enable-manager turns on ComfyUI-Manager (the Extensions button).
    comfy = start("comfyui", [sys.executable, f"{COMFY_DIR}/main.py", "--listen", "127.0.0.1", "--port", str(PORT),
                              "--enable-manager", "--enable-cors-header", public_url, "--preview-method", "auto"],
                  COMFY_LOG, cwd=COMFY_DIR)
    print("Starting ComfyUI", end="", flush=True)
    t0 = time.time()
    while not url_ok(f"{LOCAL_URL}/api/system_stats", timeout=3):
        if comfy.poll() is not None or time.time() - t0 > 600:
            raise RuntimeError("ComfyUI did not start. End of its log:\n" + tail(COMFY_LOG, 60))
        print(".", end="", flush=True)
        time.sleep(3)
    print(f" ok ({time.time() - t0:.0f} s)")

    print("Checking the link", end="", flush=True)        # open it once the way a browser does
    for _ in range(30):
        if url_ok(public_url + "/", headers={"Sec-Fetch-Site": "cross-site", "Sec-Fetch-Mode": "navigate"}):
            print(" ok\n")
            break
        print(".", end="", flush=True)
        time.sleep(3)
    else:
        print("\nThe link doesn't answer yet - give it 30 seconds before opening it.\n")

    # ---------------------------------------------------------------------------------------------
    # 4.3  The link (it opens the Qwen-Image 2.1 workflow), then ComfyUI's live log
    # ---------------------------------------------------------------------------------------------
    workflow_url = f"{public_url}/?template={TEMPLATE_NAME}&source={TEMPLATE_SOURCE}"
    print("Open ComfyUI - Qwen-Image 2.1 workflow:\n" + workflow_url)
    print("Keep this cell running while you work. Anyone with the link can use your ComfyUI; do not share it.\n")
    print("ComfyUI live log (press ■ to stop ComfyUI):\n")
    with open(COMFY_LOG, errors="replace", newline="") as log:   # newline="": keep \r so progress bars update in place
        log.seek(0, os.SEEK_END)
        while True:
            text = log.read()
            if text:
                print(text, end="", flush=True)
            elif comfy.poll() is not None:
                print("\nComfyUI has stopped - run this cell again to restart it.")
                break
            time.sleep(1)
except KeyboardInterrupt:
    print("\nStopped by you.")
finally:
    stop_all()
    print("ComfyUI and the Cloudflare link are off. Run this cell again to start them.")

## 5 · Upload inputs, save work and end the session

### Images, reference images and videos

The browser runs on your computer; ComfyUI runs on Colab's Linux machine. A local path such as `C:\Pictures\example.png` does not exist on that machine.

* **Image or reference image:** use **choose file to upload** in a `Load Image` node. Select the local file and confirm its preview and filename. ComfyUI stores it under `/content/ComfyUI/input/`. Connect the image to the input required by the loaded workflow.
* **Video:** use the upload control of the workflow's video loader, or open Colab's **Files** pane, navigate to `/content/ComfyUI/input/`, and upload there. Refresh the loader's file list, then select the uploaded name. A path-based loader needs the remote `/content/ComfyUI/input/filename.mp4` path. Its node package and the model's video support must also be present.
* **Workflow JSON:** use ComfyUI's normal **Open** command or drag the JSON onto its canvas. A JSON workflow describes the graph; it does not include model weights, custom nodes or input media. Replace missing input filenames with the files you uploaded.
* An *initial* image, *reference* image/video and control map are roles assigned by the graph. Uploading the same media does not make those roles interchangeable. The starter here is text-to-image; use the lesson's matched workflow for other operations.

### Keep individual files or the whole result folder

1. Wait for the current generation to finish. Download an image/video from ComfyUI's result controls. You can also use the file's **Download** action in Colab's Files pane. Complete your browser's **Save As** dialog if it appears, and reopen the downloaded file locally.
2. Save your edited graph with **Graph → Save As**, then **Graph → Export** for a local workflow JSON. The runtime copy appears under `/content/ComfyUI/user/default/workflows/`; export preserves the graph on your own computer.
3. For a whole folder, save the graph, stop step 4 with **■**, wait for its cleanup message, and run step 5 below. It creates one ZIP containing the selected output folder and saved workflow JSON files. Include uploaded inputs only if you want them in the backup. Use the ZIP's **Download** action in Colab's Files pane under `/content/exports/`. Model weights and notebook credentials are excluded.
4. Save the notebook in Drive and use **File → Download → Download .ipynb** for a local copy. This is separate from downloading your results.

### What survives?

| Item | Reload browser / restart step 4 in the same runtime | Runtime deleted, replaced or expired |
|---|---|---|
| Notebook saved in Drive or downloaded locally | Preserved | Preserved |
| `/content/ComfyUI` models, inputs, outputs and saved workflows | Normally remain while that runtime exists | Lost unless separately downloaded/copied |
| Files downloaded to your computer | Preserved | Preserved |
| Cloudflare link | Use the active link; restarting step 4 makes a new one | Old link no longer works |

Reopen edited graphs from **Workflows** after restarting step 4: the template link opens the starter again. A new runtime needs install/download/start again and re-uploaded inputs. Saving the notebook to Drive does **not** back up `/content`. This notebook does not mount Drive or continuously synchronize files.

When finished, confirm your local files open, stop step 4 and use **Runtime → Disconnect and delete runtime** (or terminate that session in **Manage sessions**). Closing a browser tab or merely disconnecting is not the same as releasing its GPU. Download before changing GPU type as well.

### Troubleshooting

* A later cell stays queued: stop the long-running step 4 first, run the other cell, then restart step 4.
* Cloudflare error: confirm step 4 is running and use its current link; allow a brief connection delay and reload. Do not share the temporary public link.
* Missing model or node: check the exact graph, file family, destination and required node package. A downloaded model in the wrong folder will not appear in the expected loader.
* Empty loader list after upload: refresh the list or reload ComfyUI and select the remote filename; inspect the image/video preview before generation.
* Text in a generated image: quote it in your prompt. Official Qwen-Image 2.1 templates may also need the optional prompt enhancer from step 2; on T4 use the w4a8 encoder.


In [ ]:
#@title 5 · Download output folder and saved workflows as a ZIP (after stopping step 4)
#@markdown Save your workflow and finish generation first. Stop step 4 before running this cell.
#@markdown Leave the folder blank for all outputs, or enter a relative folder such as **Qwen-Image-2.1**.
OUTPUT_SUBFOLDER = "" #@param {type:"string"}
INCLUDE_INPUTS = False #@param {type:"boolean"}

from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, uuid, zipfile

def build_result_archive(comfy_root, export_root, output_subfolder="", include_inputs=False):
    root = Path(comfy_root)
    if root.is_symlink() or not root.is_dir():
        raise ValueError("ComfyUI folder is missing or is a symbolic link; run the install cell first.")
    root = root.resolve()
    relative = Path(output_subfolder.strip() or ".")
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("Use a relative output folder, without '..'.")
    output_base = root / "output"
    output = output_base / relative
    for component in (output, *output.parents):
        if component == root: break
        if component.is_symlink(): raise ValueError("Output path contains a symbolic link.")
    if not output.resolve().is_relative_to(output_base.resolve()) or not output.is_dir():
        raise ValueError("That output folder does not exist. Check it in the Files pane.")
    sources = [(output, "output", False), (root / "user/default/workflows", "workflows", True)]
    if include_inputs: sources.append((root / "input", "input", False))
    selected, skipped = [], []
    for folder, label, json_only in sources:
        if folder.is_symlink() or not folder.resolve().is_relative_to(root):
            raise ValueError(f"Refusing linked or outside folder: {folder}")
        if not folder.exists(): continue
        for current, dirs, names in os.walk(folder, followlinks=False):
            current = Path(current)
            for name in list(dirs):
                if (current / name).is_symlink():
                    skipped.append(str((current / name).relative_to(root))); dirs.remove(name)
            for name in sorted(names):
                path = current / name
                if path.is_symlink():
                    skipped.append(str(path.relative_to(root))); continue
                if not path.is_file() or not path.resolve().is_relative_to(root): continue
                if json_only and path.suffix.lower() != ".json": continue
                if name.startswith(".") or name.endswith((".part", ".tmp")): continue
                selected.append((path, str(Path(label) / path.relative_to(folder))))
    if not any(name.startswith("output/") for _, name in selected):
        raise ValueError("No completed output files found. Generate and save a result first.")
    destination = Path(export_root)
    if destination.is_symlink(): raise ValueError("Export folder cannot be a symbolic link.")
    destination.mkdir(parents=True, exist_ok=True)
    size = sum(p.stat().st_size for p, _ in selected)
    if shutil.disk_usage(destination).free < size * 1.05 + 64 * 1024**2:
        raise RuntimeError("Not enough space for the ZIP. Download individual files or select a smaller output folder.")
    name = 'ComfyUI_results_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8] + '.zip'
    archive = destination / name
    manifest = {"output_subfolder": str(relative), "include_inputs": include_inputs,
                "files": [{"path": n, "bytes": p.stat().st_size} for p,n in selected], "skipped_links": skipped}
    with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_STORED) as z:
        for path, member in selected: z.write(path, member)
        z.writestr("manifest.json", json.dumps(manifest, indent=2))
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None: raise RuntimeError("ZIP verification failed; do not rely on this archive.")
    print(f"Saved {len(selected)} files ({archive.stat().st_size / 1e6:.1f} MB): {archive}")
    if skipped: print(f"Skipped {len(skipped)} symbolic links; see manifest.json.")
    return archive

archive = build_result_archive('/content/ComfyUI', '/content/exports', OUTPUT_SUBFOLDER, INCLUDE_INPUTS)
print("Open Colab Files, expand /content/exports, then choose Download on the ZIP shown above.")
print('Complete the browser download, then open the ZIP locally and inspect its files before deleting the runtime.')
